# MP2 Part 1: OSS Activity Retrieval

**NetID:** jvijayak  
This notebook retrieves the Part 1 data for the ten assigned projects. Run it in Google Colab or a Hydra environment, then commit this notebook and `jvijayak_project_summary.csv` to the fork.

## GitHub metadata

Collected from the GitHub REST API on 2026-10-07. The code below refreshes the values when run.

| WoC project | GitHub repository | Stars | Forks | Last pushed commit |
|---|---|---:|---:|---|
| luispedro_mahotas | luispedro/mahotas | 890 | 150 | 2026-09-23 |
| volkamerlab_opencadd | volkamerlab/opencadd | 111 | 26 | 2025-04-16 |
| ecshackweek_impedance.py | ECSHackWeek/impedance.py | 299 | 124 | 2026-04-10 |
| cesiumgs_3d-tiles | CesiumGS/3d-tiles | 2614 | 504 | 2026-09-30 |
| kbarbary_sep | kbarbary/sep | 210 | 68 | 2025-02-18 |
| cannylab_tsne-cuda | CannyLab/tsne-cuda | 1958 | 138 | 2026-07-22 |
| jmchandonia_coral | jmchandonia/CORAL | 8 | 3 | 2026-10-01 |
| mouseland_rastermap | MouseLand/rastermap | 257 | 56 | 2025-04-04 |
| sentinel-hub_sentinelhub-py | sentinel-hub/sentinelhub-py | 909 | 255 | 2026-09-16 |
| grimbough_rhdf5lib | grimbough/Rhdf5lib | 7 | 14 | 2026-06-16 |

In [ ]:
# Run in Colab/Hydra if these packages are not already installed.
%pip install -q -U python-woc pandas requests tqdm

In [ ]:
from datetime import datetime, timezone
import time
import pandas as pd
import requests
from tqdm.auto import tqdm
from woc.remote import WocMapsRemote

NETID = 'jvijayak'
PROJECTS = {
    'luispedro_mahotas': 'luispedro/mahotas',
    'volkamerlab_opencadd': 'volkamerlab/opencadd',
    'ecshackweek_impedance.py': 'ECSHackWeek/impedance.py',
    'cesiumgs_3d-tiles': 'CesiumGS/3d-tiles',
    'kbarbary_sep': 'kbarbary/sep',
    'cannylab_tsne-cuda': 'CannyLab/tsne-cuda',
    'jmchandonia_coral': 'jmchandonia/CORAL',
    'mouseland_rastermap': 'MouseLand/rastermap',
    'sentinel-hub_sentinelhub-py': 'sentinel-hub/sentinelhub-py',
    'grimbough_rhdf5lib': 'grimbough/Rhdf5lib',
}
BATCH_SIZE = 10  # below the assignment's limit of 50
WAIT_SECONDS = 1


In [ ]:
# Retrieve GitHub stars, forks, and the date of the newest commit.
github_rows = []
for project_wocid, repo in PROJECTS.items():
    response = requests.get(f'https://api.github.com/repos/{repo}', headers={'Accept': 'application/vnd.github+json'}, timeout=30)
    response.raise_for_status()
    data = response.json()
    latest = requests.get(f'https://api.github.com/repos/{repo}/commits', params={'per_page': 1}, headers={'Accept': 'application/vnd.github+json'}, timeout=30)
    latest.raise_for_status()
    github_rows.append({
        'project_wocid': project_wocid,
        'github_repository': repo,
        'stars': data['stargazers_count'],
        'forks': data['forks_count'],
        'last_commit_date': latest.json()[0]['commit']['committer']['date'][:10],
    })
github_df = pd.DataFrame(github_rows)
display(github_df)

In [ ]:
# Retrieve each project's commit SHA-1 values from WoC, then cache them locally.
woc = WocMapsRemote(base_url='https://worldofcode.org/api/')
project_commits = {}
for project_wocid in tqdm(PROJECTS, desc='WoC project-to-commit lookups'):
    project_commits[project_wocid] = list(woc.get_values('p2c', project_wocid))
    time.sleep(WAIT_SECONDS)

sha_df = pd.DataFrame(
    [(project, sha) for project, shas in project_commits.items() for sha in shas],
    columns=['project_wocid', 'commit_sha1'],
)
sha_df.to_csv(f'{NETID}_commit_sha1s.csv', index=False, sep=';')
display(sha_df.groupby('project_wocid').size().rename('number_of_commits').to_frame())

In [ ]:
# Retrieve commit contents in batches of 10 and preserve every project association.
rows = []
errors = []
for project_wocid, shas in project_commits.items():
    for start in tqdm(range(0, len(shas), BATCH_SIZE), desc=project_wocid):
        batch = shas[start:start + BATCH_SIZE]
        result, batch_errors = woc.get_values_many('commit.tch', batch)
        if batch_errors:
            errors.extend((project_wocid, error) for error in batch_errors)
        for sha, values in result.items():
            commit = values[0]  # remote client wraps the commit object once
            rows.append({
                'project_wocid': project_wocid,
                'commit_sha1': sha,
                'author': commit[2][0],
                'time': int(commit[2][1]),
                'commit message': commit[4],
            })
        time.sleep(WAIT_SECONDS)

if errors:
    print(f'{len(errors)} WoC batch errors recorded; rerun the affected project batches before submitting.')
else:
    print('All WoC batches returned without reported errors.')

In [ ]:
# Write the Part 1 deliverable using the exact required column order and semicolon separator.
summary_df = pd.DataFrame(rows, columns=['project_wocid', 'commit_sha1', 'author', 'time', 'commit message'])
summary_path = f'{NETID}_project_summary.csv'
summary_df.to_csv(summary_path, index=False, sep=';')

assert list(summary_df.columns) == ['project_wocid', 'commit_sha1', 'author', 'time', 'commit message']
assert not summary_df.empty, 'No commit rows were retrieved.'
assert summary_df['project_wocid'].nunique() == len(PROJECTS), 'At least one assigned project has no rows.'
print(f'Wrote {len(summary_df):,} rows to {summary_path}')
display(summary_df.head())

In [ ]:
# Required WoC report: commit count, author count, maximum time, and minimum time for every project.
woc_stats = (summary_df.groupby('project_wocid', as_index=False)
    .agg(number_of_commits=('commit_sha1', 'size'),
         number_of_authors=('author', 'nunique'),
         max_time=('time', 'max'),
         min_time=('time', 'min')))
woc_stats['max_time_utc'] = pd.to_datetime(woc_stats['max_time'], unit='s', utc=True)
woc_stats['min_time_utc'] = pd.to_datetime(woc_stats['min_time'], unit='s', utc=True)
part1_report = woc_stats.merge(github_df, on='project_wocid', how='left')
part1_report.to_csv(f'{NETID}_part1_metadata.csv', index=False, sep=';')
display(part1_report)

## Submission check

After the cells complete, commit `jvijayak.ipynb`, `jvijayak_project_summary.csv`, and the optional cached SHA and metadata CSV files to the fork. The summary CSV is the required Part 1 deliverable; the other two CSVs make the retrieval auditable and easier to resume.